# 05 · Attack-class evaluation & acceptance gates (Alert Triage #01)
An aggregate ROC of 0.99 can still hide an attack class the model never catches, so recall is checked per class. Then the plan §8 gates are applied.

Logic: `src/evaluation/attack_class_evaluator.py` and `src/evaluation/operating_point_validator.py`.

In [ ]:
import sys
from pathlib import Path
SLOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(SLOT))
from src.config import load_all_configs
from src import data_source as ds
cfg = load_all_configs()
mcfg, fcfg = cfg['model'], cfg['feature']
selected = ds.load_selected(mcfg)['features']
print('dataset', mcfg['data']['dataset'], '| model features', selected)


## Recall per attack class — the blind-spot check

In [ ]:
import json, pandas as pd
m = json.load(open(SLOT/'outputs'/'metrics.json'))
pc = pd.DataFrame(m['per_class']).sort_values('recall')
pc.plot.barh(x='attack_class', y='recall', figsize=(9,5), color='#2a78d6',
             title='Detection recall per attack class', legend=False);
pc

## The weakest class is what drift monitoring watches

In [ ]:
print(m['weakest_attack_class'])

## Acceptance gates (plan §8)

In [ ]:
g = m['acceptance']
for k, v in g.items():
    print(f'{k:34s} {v}')

## Queue quality — analysts work the top of the ranking

In [ ]:
print('NDCG@100        ', m['ndcg_at_100'])
print('precision@1000  ', m['precision_at_1000'])
print('lift@1000       ', m['lift_at_1000'], 'x random')